<a href="https://colab.research.google.com/github/AishaniiiRoy/Movie_Recommendation/blob/main/Movie_Recommendation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded=files.upload()
import pandas as pd
import random


Saving movies_dataset_unique.csv to movies_dataset_unique (5).csv


In [ ]:
movies_df=pd.read_csv("movies_dataset_unique.csv")

In [ ]:
movies_df[0:5]

,movieId,title,genres
0,1,Toy Story (1995),Animation|Children|Comedy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [ ]:
movies_df.drop( 'genres', axis = 1, inplace = True )

In [ ]:
from google.colab import files
uploaded=files.upload()

Saving ratings_dataset.csv to ratings_dataset (2).csv


In [ ]:
rating_df=pd.read_csv("ratings_dataset.csv")

In [ ]:
rating_df[0:5]

,userId,movieId,rating
0,39,317,1
1,29,451,5
2,15,123,2
3,43,390,4
4,8,9,2


In [ ]:
def find_common_movies(user1, user2):
    common_movies = rating_df[rating_df.userId == user1].merge(
        rating_df[rating_df.userId == user2],
        on="movieId",
        suffixes=('_x', '_y')
    )
    return common_movies.merge(movies_df, on="movieId")


In [ ]:
common_movies = find_common_movies(1,2)
print(type(common_movies))


<class 'pandas.core.frame.DataFrame'>


In [ ]:
common_movies[(common_movies.rating_x >= 1.0) &
((common_movies.rating_y >= 1.0))]

,userId_x,movieId,rating_x,userId_y,rating_y,title
0,1,108,4,2,4,Movie 108 (2000)
1,1,293,3,2,2,Movie 293 (2000)
2,1,390,4,2,2,Movie 390 (2000)


In [ ]:
common_movies = find_common_movies( 1, 2 )
common_movies

,userId_x,movieId,rating_x,userId_y,rating_y,title
0,1,108,4,2,4,Movie 108 (2000)
1,1,293,3,2,2,Movie 293 (2000)
2,1,390,4,2,2,Movie 390 (2000)


In [ ]:
from sklearn.metrics import pairwise_distances
import numpy as np

# Ensure no duplicate userId–movieId pairs (keep highest rating)
rating_df = (
    rating_df
    .sort_values('rating', ascending=False)
    .drop_duplicates(subset=['userId', 'movieId'])
)

# Create user–movie rating matrix
rating_mat = rating_df.pivot(
    index='movieId',
    columns='userId',
    values='rating'
).fillna(0)   # directly fill NaNs here

# Compute correlation-based similarity (1 - distance)
movie_sim = 1 - pairwise_distances(rating_mat.values, metric="correlation")

# Convert to DataFrame with movieId index
movie_sim_df = pd.DataFrame(
    movie_sim,
    index=rating_mat.index,
    columns=rating_mat.index
)

# Fill diagonal with 0 (ignore self-similarity)
np.fill_diagonal(movie_sim_df.values, 0)

print(movie_sim_df.head())


movieId       1         2         3         4         5         7         8    \
movieId                                                                         
1        0.000000  0.220222  0.363211 -0.072184 -0.083952 -0.081592  0.027113   
2        0.220222  0.000000 -0.078562 -0.060624 -0.070507 -0.068525 -0.091083   
3        0.363211 -0.078562  0.000000 -0.067101 -0.057718 -0.075847 -0.037806   
4       -0.072184 -0.060624 -0.067101  0.000000  0.441623  0.340529  0.408428   
5       -0.083952 -0.070507 -0.057718  0.441623  0.000000  0.241339 -0.040213   

movieId       9         10        11   ...       491       492       493  \
movieId                                ...                                 
1       -0.061302  0.241434  0.048318  ... -0.064075 -0.064752 -0.081592   
2       -0.051484  0.106363 -0.046377  ... -0.053813 -0.054382 -0.068525   
3       -0.056985  0.224434 -0.051333  ... -0.059563  0.221961 -0.075847   
4       -0.043974 -0.066463 -0.039612  ... -0.045963

In [ ]:
movie_sim_df.iloc[0:5, 0:5]

movieId,1,2,3,4,5
movieId,,,,,
1,0.000000,0.220222,0.363211,-0.072184,-0.083952
2,0.220222,0.000000,-0.078562,-0.060624,-0.070507
3,0.363211,-0.078562,0.000000,-0.067101,-0.057718
4,-0.072184,-0.060624,-0.067101,0.000000,0.441623
5,-0.083952,-0.070507,-0.057718,0.441623,0.000000


In [ ]:
movie_sim_df.shape

(492, 492)

In [ ]:
def get_similar_movies(movieid, top_n=5):
    # find the index of the given movieId
    movieidx = movies_df[movies_df.movieId == movieid].index[0]

    # get similarity scores for that movie
    sim_scores = movie_sim_df.iloc[movieidx]

    # sort scores descending, get top N
    top_movies = sim_scores.sort_values(ascending=False).head(top_n)

    # join with movie titles
    return movies_df.loc[top_movies.index, ["movieId", "title"]]


In [ ]:
movies_df[movies_df.movieId == 100]

,movieId,title
99,100,Movie 100 (2000)


In [ ]:
movies_df[movies_df.movieId == 20]

,movieId,title
19,20,Money Train (1995)


In [ ]:
get_similar_movies(20)


,movieId,title
movieId,,
307,308,Movie 308 (2000)
276,277,Movie 277 (2000)
331,332,Movie 332 (2000)
91,92,Movie 92 (2000)
372,373,Movie 373 (2000)
